<a href="https://colab.research.google.com/github/divyansh212/LLM-Models-from-Scratch-/blob/main/LLM_from_scratch(Andrej_KARAPATHY)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-07 06:13:58--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-03-07 06:13:59 (36.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F
with open ('input.txt','r',encoding='utf-8') as f:
  text=f.read()
chars=sorted(list(set(text)))
vocab_size=len(chars)
stoic={}
itos={}
for i,ch in enumerate(chars):
  stoic[ch]=i
for i,ch in enumerate(chars):
  itos[i]=ch
encode=lambda s:[stoic[c] for c in s ]#convert query into numbers
decode =lambda l: ''.join([itos[i] for i in l])#convert solution token into string
data=torch.tensor(encode(text),dtype=torch.long)
n=int(0.9*len(data))
test_data=data[n:]
train_data=data[:n]
block_size=8
batch_size=32
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
xb, yb = get_batch('train')
class BRLM(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding=nn.Embedding(vocab_size,vocab_size)
  def forward(self,idx,target=None):
    logits=self.token_embedding(idx)
    if target is None:
      loss=None
    else:
      b,t,c=logits.shape
      logits=logits.view(b*t,c)
      target=target.view(b*t)
      loss=F.cross_entropy(logits,target)
    return logits, loss
  def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
model=BRLM(vocab_size)
logits,loss=model(xb,yb)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)
for epoch in range(10000):
  xb,yb=get_batch('train')
  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
x=decode(model.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())
b,t,c=4,8,2
x=torch.randn(b,t,c)
xbow=torch.zeros(b,t,c)
for n in range(b):
  for r in range(t):
    xprev=x[n,:r+1]
    xbow[n,r]=torch.mean(xprev,0)
w=torch.tril(torch.ones(t,t))
w=w/w.sum(1,keepdim=True)
xbow2=w @ x
torch.allclose(xbow,xbow2)

True